# Treino por reforço da IA do GMS (Qwen3 4B) - Colab / Kaggle

Este caderno ajusta o modelo que escreve SQL para as perguntas da fábrica. Roda numa GPU T4 (Colab grátis ou Kaggle).
Fluxo: **SFT** (aprende o formato e os exemplos aprovados) -> **DPO** (prefere o SQL certo ao erro conhecido) ->
**GRPO** (treino por reforço: o modelo escreve SQL, o SQL roda nos dados e ganha nota) -> avaliação antes e depois -> **GGUF** para o Ollama.

**APIs conferidas na documentação atual em 01/11/2026** (Unsloth e TRL, Hugging Face):
- Unsloth: `FastLanguageModel.from_pretrained(..., load_in_4bit=True)`, `FastLanguageModel.get_peft_model(...)`,
  `model.save_pretrained_gguf(pasta, tokenizer, quantization_method="q4_k_m")`.
- TRL: `SFTTrainer/SFTConfig`, `DPOTrainer/DPOConfig`, `GRPOTrainer/GRPOConfig` (`num_generations`, `max_completion_length`).
- Função de recompensa do TRL: `f(prompts, completions, **kwargs) -> list[float]`; as colunas extras do dataset chegam em `kwargs`;
  no formato conversacional a resposta está em `completion[0]["content"]` (aqui usamos texto simples, então `completion` já é string).
- As páginas de documentação **não fixam número de versão** (usam `pip install unsloth`). A célula 2 imprime as versões que foram instaladas
  nesta execução; anote-as aqui se o treino funcionar: Unsloth ____ / TRL ____ / transformers ____.
- Se uma versão nova mudar um argumento, a função `cfg` da célula 5 ignora o argumento que a classe não aceita e avisa.

**Antes:** `python exportar_treino.py` no seu PC gera o `pacote_treino.zip`. Ligue a GPU (Ambiente de execução -> T4).

## 1. Instalar as bibliotecas

In [ ]:
!pip install -q unsloth trl duckdb pandas pyarrow

## 2. Versões instaladas e GPU

In [ ]:
import importlib.metadata as md_
import torch
for pacote in ("unsloth", "trl", "transformers", "peft", "duckdb", "torch"):
    try:
        print(pacote, md_.version(pacote))
    except Exception:
        print(pacote, "?")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA - ligue a GPU T4")

## 3. Subir o pacote_treino.zip
No Colab abre a janela de upload. No Kaggle, adicione o zip como Dataset e ele é achado em `/kaggle/input`.

In [ ]:
import glob, os, zipfile
KAGGLE = os.path.exists("/kaggle")
BASE = "/kaggle/working" if KAGGLE else "/content"
try:
    from google.colab import files
    enviado = files.upload()
    zip_path = next(iter(enviado))
except ImportError:
    zip_path = next(iter(glob.glob("/kaggle/input/**/pacote_treino.zip", recursive=True)), "pacote_treino.zip")
PASTA = os.path.join(BASE, "treino")
os.makedirs(PASTA, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(PASTA)
print(os.listdir(PASTA))

## 4. Carregar os dados e a conexão do DuckDB
`recompensa.py` e `gms_ia.py` vieram dentro do zip: a guarda de SQL (só leitura) é a **mesma** do gms_ia.
A conexão só enxerga os `.parquet` do cache.

In [ ]:
import json, sys
sys.path.insert(0, PASTA)
os.chdir(PASTA)
import recompensa

def ler(nome):
    p = os.path.join(PASTA, nome)
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()] if os.path.exists(p) else []

sft, dpo, grpo, ev = ler("sft.jsonl"), ler("dpo.jsonl"), ler("grpo.jsonl"), ler("eval.jsonl")
print(f"sft {len(sft)} | dpo {len(dpo)} | grpo {len(grpo)} | eval {len(ev)}")
if len(sft) < 100: print("AVISO: poucos exemplos de SFT.")
if len(ev) < 30: print("AVISO: poucas perguntas de avaliação; o placar oscila.")
CON = recompensa.abrir(os.path.join(PASTA, "dados"))

## 5. Carregar o Qwen3 4B em 4 bits com LoRA
4 bits cabe na T4 (16 GB). LoRA treina só um pedaço pequeno do modelo. O prompt do sistema é grande (esquema + mapa),
por isso o limite de prompt é 2200 tokens (a T4 não aguenta mais no GRPO); exemplos maiores são descartados e a célula conta quantos. Se descartar muitos, enxugue o `mapa_dados.md`.

In [ ]:
import dataclasses
import torch
from unsloth import FastLanguageModel

MAX_SEQ = 2700      # cabe o GRPO na T4 (prompt + resposta de ate 400 tokens)
MAX_PROMPT = 2200   # exemplos com prompt maior sao descartados (a celula avisa quantos)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

assert tokenizer.eos_token == "<|im_end|>", f"eos inesperado: {tokenizer.eos_token}"  # o TRL acrescenta o EOS sozinho

import gc

def liberar_gpu(*nomes):
    for n in nomes:
        globals().pop(n, None)
    gc.collect()
    torch.cuda.empty_cache()

def cfg(Classe, **kw):
    """Cria a config do TRL ignorando argumentos que a versao instalada nao conhece."""
    aceitos = {f.name for f in dataclasses.fields(Classe)}
    for k in [k for k in kw if k not in aceitos]:
        print(f"(aviso) {Classe.__name__} nao tem '{k}' nesta versao; ignorei.")
        kw.pop(k)
    return Classe(**kw)

def texto_prompt(mensagens):
    # mesmo molde do Ollama com think desligado (o gms_ia manda think=False)
    return tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True, enable_thinking=False)

## 6. Avaliação (antes do treino)
Para cada pergunta do `eval.jsonl` (que nunca entra no treino) o modelo escreve o SQL, o SQL roda no DuckDB e
conferimos com o esperado (números com tolerância de 1%). Taxa de acerto = **resultado certo**, não texto igual.

In [ ]:
import torch

def avaliar_modelo(itens, max_novos=400):
    FastLanguageModel.for_inference(model)
    certos = executam = 0
    for it in itens:
        entrada = tokenizer(texto_prompt(it["prompt"]), return_tensors="pt").to("cuda")
        with torch.no_grad():
            saida = model.generate(**entrada, max_new_tokens=max_novos, do_sample=False)
        texto = tokenizer.decode(saida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True)
        r = recompensa.analisar(CON, recompensa.texto_da(texto), it)
        certos += r["bate"]
        executam += r["executa"]
    n = max(len(itens), 1)
    print(f"acerto por execucao: {certos}/{len(itens)} ({100 * certos / n:.0f}%) | executam sem erro: {executam}/{len(itens)}")
    return certos, executam

ANTES = avaliar_modelo(ev) if ev else (0, 0)

## 7. Etapa SFT (aprender o formato e os exemplos aprovados)
O dataset vira texto já no molde do Qwen3 (prompt) e o SQL em bloco (completion); a perda conta só a resposta.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def para_texto(item):
    return {"prompt": texto_prompt(item["prompt"]),
            "completion": item["completion"][0]["content"]}

dados_sft = [para_texto(x) for x in sft]
n0 = len(dados_sft)
dados_sft = [d for d in dados_sft if len(tokenizer(d["prompt"]).input_ids) <= MAX_PROMPT]
print("SFT usa", len(dados_sft), "exemplos (descartados por prompt grande:", n0 - len(dados_sft), ")")
if dados_sft:
    FastLanguageModel.for_training(model)
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=Dataset.from_list(dados_sft),
        args=cfg(SFTConfig,
                 per_device_train_batch_size=1, gradient_accumulation_steps=8,
                 num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="linear", warmup_steps=5,
                 logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
                 max_length=MAX_SEQ, max_seq_length=MAX_SEQ, output_dir="saida_sft", report_to="none", seed=3407),
    )
    trainer.train()
    del trainer
else:
    print("Sem exemplos de SFT; etapa pulada.")
liberar_gpu("trainer")

## 8. Etapa DPO (prefere o SQL certo ao erro conhecido)
Pula sozinha se o `dpo.jsonl` estiver vazio.

In [ ]:
from trl import DPOTrainer, DPOConfig

dados_dpo = [{"prompt": texto_prompt(x["prompt"]),
              "chosen": x["chosen"][0]["content"],
              "rejected": x["rejected"][0]["content"]} for x in dpo]
dados_dpo = [d for d in dados_dpo if len(tokenizer(d["prompt"]).input_ids) <= MAX_PROMPT]
if dados_dpo:
    print("DPO usa", len(dados_dpo), "pares")
    FastLanguageModel.for_training(model)
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        processing_class=tokenizer,
        train_dataset=Dataset.from_list(dados_dpo),
        args=cfg(DPOConfig,
                 beta=0.1, per_device_train_batch_size=1, gradient_accumulation_steps=8,
                 num_train_epochs=1, learning_rate=5e-6, optim="adamw_8bit", logging_steps=5,
                 max_length=MAX_SEQ, max_prompt_length=MAX_PROMPT, output_dir="saida_dpo", report_to="none", seed=3407),
    )
    dpo_trainer.train()
    del dpo_trainer
else:
    print("Sem pares de DPO; etapa pulada.")
liberar_gpu("dpo_trainer")

## 9. Funções de recompensa do GRPO
Cada resposta do modelo ganha pontos somando 4 funções:
- **formato** +0,1: SQL dentro de um bloco ```sql```;
- **só leitura** 0 se passar na guarda do `gms_ia` (só `SELECT`/`WITH`, sem palavras proibidas), **-1** se não passar;
- **executa** +0,2: roda no DuckDB sem erro dentro do limite de tempo (10 s);
- **resultado** +1,0: bate com o esperado (números com tolerância de 1%, ou o resultado do SQL de referência) e tem a mesma forma dele (mesmo nº de linhas, até 2x as células). Exige `FROM` numa tabela do cache e resultado vazio nunca vale.

Assinatura do TRL: `f(completions, **kwargs)`; as colunas `pergunta`, `esperado` e `ref_json` do dataset chegam em `kwargs`.

In [ ]:
def _analises(completions, **kw):
    saida = []
    for i, c in enumerate(completions):
        item = {k: kw[k][i] for k in ("pergunta", "esperado", "ref_json") if k in kw}
        saida.append(recompensa.analisar(CON, recompensa.texto_da(c), item))
    return saida

def recompensa_formato(completions, **kwargs):
    return [0.1 if recompensa.tem_bloco(recompensa.texto_da(c)) else 0.0 for c in completions]

def recompensa_leitura(completions, **kwargs):
    return [0.0 if a["leitura"] else -1.0 for a in _analises(completions, **kwargs)]

def recompensa_execucao(completions, **kwargs):
    return [0.2 if a["executa"] else 0.0 for a in _analises(completions, **kwargs)]

def recompensa_resultado(completions, **kwargs):
    return [1.0 if a["bate"] else 0.0 for a in _analises(completions, **kwargs)]

## 10. Etapa GRPO (treino por reforço)
A T4 é lenta: 300 passos levam de 1 a 3 horas. Reduza `max_steps` para um teste rápido.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

dados_grpo = []
for x in grpo:
    if len(tokenizer(texto_prompt(x["prompt"])).input_ids) <= MAX_PROMPT:
        dados_grpo.append({"prompt": texto_prompt(x["prompt"]), "pergunta": x["pergunta"],
                           "esperado": x["esperado"], "ref_json": x["ref_json"]})
print("GRPO usa", len(dados_grpo), "de", len(grpo), "perguntas (resto: prompt grande demais)")
import trl
print("TRL", trl.__version__)
# Lote compativel com as duas regras do TRL (versoes antigas: lote por GPU; novas: lote por GPU x acumulacao):
# per_device = num_generations = 4 e acumulacao 2 => ambos os lotes (4 e 8) sao multiplos de 4.
NUM_GER, POR_GPU, ACUM = 4, 4, 2
assert POR_GPU % NUM_GER == 0 and (POR_GPU * ACUM) % NUM_GER == 0
if dados_grpo:
    FastLanguageModel.for_training(model)
    grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[recompensa_formato, recompensa_leitura, recompensa_execucao, recompensa_resultado],
        train_dataset=Dataset.from_list(dados_grpo),
        args=cfg(GRPOConfig,
                 num_generations=NUM_GER, per_device_train_batch_size=POR_GPU, gradient_accumulation_steps=ACUM,
                 max_completion_length=400, max_prompt_length=MAX_PROMPT,
                 learning_rate=5e-6, lr_scheduler_type="cosine", warmup_ratio=0.1, optim="adamw_8bit",
                 max_steps=300, logging_steps=5, temperature=1.0, use_vllm=False,
                 output_dir="saida_grpo", report_to="none", seed=3407),
    )
    grpo_trainer.train()
    del grpo_trainer
else:
    print("Sem perguntas com resposta esperada; GRPO pulado.")
liberar_gpu("grpo_trainer")

## 11. Avaliação (depois do treino)

In [ ]:
DEPOIS = avaliar_modelo(ev) if ev else (0, 0)
n = max(len(ev), 1)
print(f"ANTES : {ANTES[0]}/{len(ev)} ({100 * ANTES[0] / n:.0f}%)")
print(f"DEPOIS: {DEPOIS[0]}/{len(ev)} ({100 * DEPOIS[0] / n:.0f}%)")
if DEPOIS[0] <= ANTES[0]:
    print("O treino nao melhorou o placar. Nao troque o modelo; junte mais exemplos e tente de novo.")

## 12. Exportar GGUF (q4_k_m) e o Modelfile do Ollama (`gms-qwen`)

In [ ]:
SAIDA_GGUF = os.path.join(BASE, "gms-qwen-gguf")
model.save_pretrained_gguf(SAIDA_GGUF, tokenizer, quantization_method="q4_k_m")
achados = sorted(glob.glob(os.path.join(BASE, "**", "*.gguf"), recursive=True))
print(achados)
gguf = next((g for g in achados if "q4_k_m" in g.lower()), achados[0] if achados else None)
assert gguf, "nenhum .gguf gerado"
nome = os.path.basename(gguf)
modelfile = f'''FROM ./{nome}
TEMPLATE """{{{{- range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
<think>

</think>

"""
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0
PARAMETER num_ctx 8192
'''
MODELFILE = os.path.join(os.path.dirname(gguf), "Modelfile")
open(MODELFILE, "w", encoding="utf-8", newline="\n").write(modelfile)
print(modelfile)

## 13. Baixar os arquivos
No PC: coloque o `.gguf` e o `Modelfile` na mesma pasta e rode `ollama create gms-qwen -f Modelfile`.

In [ ]:
try:
    from google.colab import files
    files.download(MODELFILE)
    files.download(gguf)
except ImportError:
    print("Kaggle: baixe pela aba Output (Data -> Output) os arquivos:", gguf, MODELFILE)